Oracle AI Data Platform v1.0

# HubSpot Ingestion (Full Refresh and Incremental)

Lands HubSpot **contacts**, **companies**, **deals** and their **associations** in Delta tables in the catalog and schema named in `hubspot_ingest.yaml`, using the `aidp-connector-hubspot` package. Read-only against HubSpot.

- **Incremental (default):** reads only what changed since the last successful run, using a per-object watermark kept in `hubspot_sync_state`. An object with no watermark is loaded in full.
- **Full:** lists every live and archived record and overwrites the tables.

**Before you run**

1. Create a HubSpot service key, or a legacy private app token, with the scopes `crm.objects.contacts.read`, `crm.objects.companies.read` and `crm.objects.deals.read`.
2. Store the token in the AIDP Credential Store and set `hubspot.credential_name` in the config.
3. Install the `aidp-connector-hubspot` wheel on the cluster (see step 1 below).
4. Copy `hubspot_ingest.sample.yaml` to `hubspot_ingest.yaml` in your workspace and edit the target catalog and schema.

See the package `README.md` for details and known limits.

## 1. Install the connector

Skip this cell if the wheel is already installed as a cluster library. Otherwise upload `aidp_connector_hubspot-<version>-py3-none-any.whl` to your workspace, then uncomment the line below and set its path. It also installs `requests` and `pyyaml`.

In [ ]:
# %pip install /Workspace/<path-to>/aidp_connector_hubspot-0.1.0-py3-none-any.whl

In [ ]:
from aidp_connector_hubspot import (
    __version__,
    format_summary,
    load_config,
    raise_on_failure,
    read_token,
    run,
)

print("aidp-connector-hubspot", __version__)

## 2. Load the configuration

In [ ]:
# Your copy of hubspot_ingest.sample.yaml.
CONFIG_PATH = "/Workspace/<path-to>/hubspot_ingest.yaml"

config = load_config(CONFIG_PATH)

print("target :", config.target.qualified_schema)
print("objects:", ", ".join(config.sync.objects))
print("mode   :", config.sync.mode)

## 3. Optional per-run override

When this notebook runs as a job, a job parameter `MODE` (`incremental` or `full`) overrides `sync.mode` for that run only. Interactive runs use the config.

In [ ]:
_MISSING = "\x00__NO_PARAMETER__"


def job_parameter(name):
    """Read a job parameter, or None. Names are case-sensitive on the platform,
    so try the usual spellings: whoever registers the job may write 'mode', not 'MODE'."""
    for candidate in dict.fromkeys((name, name.upper(), name.lower())):
        try:
            # oidlUtils is provided by the AIDP runtime - do not import it.
            value = oidlUtils.parameters.getParameter(candidate, _MISSING)  # noqa: F821
        except NameError:
            return None  # not running on AIDP
        if not (value == _MISSING or value is None or str(value).strip() == ""):
            return str(value).strip()
    return None


MODE = job_parameter("MODE")
print("mode override:", MODE or "(none)")

## 4. Read the HubSpot token

In [ ]:
try:
    get_secret = aidputils.secrets.get  # noqa: F821 - provided by the AIDP runtime, do not import it
except NameError:
    get_secret = None  # not on AIDP: read the token from the environment variable instead

token = read_token(config.hubspot, get_secret)
print("token read (not shown)")

## 5. Run the sync

Each object is synced on its own: one failing does not stop the others, and the last line raises if any failed. Associations are read from the ids in the `contacts` and `deals` tables, so they run last.

In [ ]:
summaries = run(spark, config, token, mode_override=MODE)  # noqa: F821 - spark is provided by the notebook

print(format_summary(summaries))
raise_on_failure(summaries)

## 6. Look at the result

In [ ]:
prefix = config.target.table_prefix
schema = config.target.qualified_schema

# run() already printed hubspot_sync_state, so only show one data table here.
shown = [name for name in ("contacts", "companies", "deals") if name in config.sync.objects]
if shown:
    spark.table(f"{schema}.{prefix}{shown[0]}").show()  # noqa: F821
else:
    print("No contacts, companies or deals table is configured; nothing to show.")

More queries. Replace `<prefix>` with `target.table_prefix` (empty by default):

```sql
-- Deals by stage
SELECT deal_stage, COUNT(*) AS deals, SUM(amount) AS amount
FROM <catalog>.<schema>.<prefix>deals WHERE NOT archived GROUP BY deal_stage;

-- Contacts and the companies they belong to
SELECT c.email, co.name
FROM <catalog>.<schema>.<prefix>associations a
JOIN <catalog>.<schema>.<prefix>contacts c ON c.id = a.from_id
JOIN <catalog>.<schema>.<prefix>companies co ON co.id = a.to_id
WHERE a.from_object = 'contacts' AND a.to_object = 'companies' AND NOT c.archived AND NOT co.archived;

-- Watermarks and the outcome of the last run per object
SELECT * FROM <catalog>.<schema>.<prefix>hubspot_sync_state;
```